In [1]:
import pandas as pd
import numpy as np

# 1. 데이터 불러오기 (파일명은 실제 파일 경로에 맞게 수정하세요)
df = pd.read_csv(r"C:\Users\ADMIN\Downloads\데이터마트.csv")

# 2. 한글 기반 카테고리 분류 함수
def classify_category_ko(name):
    name = str(name).replace(" ", "").lower() # 공백 제거 및 소문자화로 매칭 확률 높임
    
    # 상세 모델부터 먼저 체크 (우선순위 고려)
    if '아이패드미니' in name or 'ipadmini' in name: return 'iPad Mini'
    elif '아이패드에어' in name or 'ipadair' in name: return 'iPad Air'
    elif '아이패드프로' in name or 'ipadpro' in name: return 'iPad Pro'
    elif '아이패드' in name or 'ipad' in name: return 'iPad'
    
    elif '맥북에어' in name or 'macbookair' in name: return 'MacBook Air'
    elif '맥북프로' in name or 'macbookpro' in name: return 'MacBook Pro'
    elif '맥북' in name or 'macbook' in name: return 'MacBook'
    
    elif '아이폰' in name or 'iphone' in name: return 'iPhone'
    elif '애플워치' in name or 'applewatch' in name: return 'Apple Watch'
    elif '맥미니' in name or 'macmini' in name: return 'Mac Mini'
    elif '아이맥' in name or 'imac' in name: return 'iMac'
    
    else: return '기타'

# 카테고리 컬럼 생성
df['category'] = df['name_buy'].apply(classify_category_ko)

# 3. 데이터 정제 및 보존율 계산
# 출시가(release_price)나 판매가(price_sell)가 0인 데이터 제외
df = df[(df['release_price'] > 0) & (df['price_sell'] > 0)].copy()

# 보존율 계산 (%)
df['retention_rate'] = (df['price_sell'] / df['release_price']) * 100

# 4. 분기(3개월 단위) 그룹화
# device_age_months가 0~2개월이면 1분기, 3~5개월이면 2분기...
df['age_quarter'] = (df['device_age_months'] // 3) + 1

# 5. 결과 집계: 카테고리별, 분기별 평균 보존율
analysis_result = df.groupby(['category', 'age_quarter'])['retention_rate'].mean().reset_index()

# 6. 보기 편하게 피벗 테이블로 변환 (행: 카테고리, 열: 분기)
pivot_df = analysis_result.pivot(index='category', columns='age_quarter', values='retention_rate')

# 결과 출력 (소수점 첫째자리까지)
print("### [한글 모델명 기준] 카테고리별/분기별 평균 보존율 (%) ###")
print(pivot_df.round(1))

# (참고) 엑셀 파일로 결과 저장하기
# pivot_df.to_excel('retention_analysis_result.xlsx')

### [한글 모델명 기준] 카테고리별/분기별 평균 보존율 (%) ###
age_quarter  0.0   1.0   2.0   3.0   4.0   5.0   6.0    7.0   8.0   9.0   ...  \
category                                                                  ...   
Apple Watch   NaN   NaN  81.8  71.4  71.8  66.5  44.8   51.8  42.5  42.7  ...   
Mac Mini      NaN   NaN   NaN   NaN  82.3  78.8  69.5   83.5   NaN  67.7  ...   
MacBook       NaN   NaN   NaN   NaN   NaN   NaN   NaN    NaN   NaN  62.0  ...   
MacBook Air   NaN  77.7  67.3  65.8  77.1  62.9  64.1   65.6  59.5  51.6  ...   
MacBook Pro   NaN  61.0  67.0   NaN  75.1  63.2  56.9   58.0  59.0  50.3  ...   
iMac          NaN   NaN   NaN   NaN  81.1  68.5  62.7   76.5  56.9  62.7  ...   
iPad          NaN   NaN   NaN  73.5  69.0  67.6  54.5   54.4  95.7  68.4  ...   
iPad Air      NaN   NaN   NaN   NaN   NaN   NaN  71.1  100.1  65.2  69.4  ...   
iPad Mini    82.2   NaN   NaN  81.7  70.3  70.3   NaN   63.4  64.7  67.4  ...   
iPad Pro      NaN  90.3  85.7  81.7  71.0  70.0  75.6   67.2  69.4  

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager, rc

# 1. 한글 폰트 설정 (Windows 기준: 맑은 고딕)
try:
    font_path = "C:/Windows/Fonts/malgun.ttf"
    font_name = font_manager.FontProperties(fname=font_path).get_name()
    rc('font', family=font_name)
except:
    print("한글 폰트 설정에 실패했습니다. (경로 확인 필요)")

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 2. 모델에서 중요도 추출 및 정렬
importances = rf_buy.feature_importances_  # 학습시킨 모델 객체명에 맞게 수정 (rf_buy 또는 rf_buy_model)
feature_names = X_b.columns               # 학습에 쓴 데이터프레임 이름

feature_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_imp_df = feature_imp_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

# 3. 시각화 (상위 10개만 출력)
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_imp_df.head(10), palette='viridis')
plt.title('매입가 결정 핵심 변수', fontsize=15)
plt.xlabel('중요도 (비중)', fontsize=12)
plt.ylabel('변수 이름', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

# 4. 누적 중요도와 함께 수치 출력
feature_imp_df['누적_중요도'] = feature_imp_df['Importance'].cumsum()
print("📊 [변수 중요도 리포트]")
print(feature_imp_df.head(15)) # 상위 15개 수치 확인

In [3]:
import pandas as pd
import numpy as np
import re
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error, r2_score

# ============================================================
# [SECTION 1] 데이터 전처리 및 계층형 DB 구축
# ============================================================

full_path = r"C:\Users\ADMIN\Downloads\데이터마트.xlsx"

def clean_numeric(value):
    if pd.isna(value): return np.nan
    try:
        s = str(value).replace(',', '').replace('원', '').strip()
        if '~' in s:
            nums = re.findall(r'\d+', s)
            return np.mean([float(n) for n in nums]) if nums else np.nan
        s = re.sub(r'[^\d.]', '', s)
        return float(s) if s else np.nan
    except: return np.nan

def extract_number(text):
    if pd.isna(text): return 0
    text = str(text).upper()
    numbers = re.findall(r'\d+', text)
    if not numbers: return 0
    val = int(numbers[0])
    if 'TB' in text: val *= 1024
    return val

def classify_category_ko(name):
    name = str(name).replace(" ", "").lower()
    if '아이패드미니' in name or 'ipadmini' in name: return 'iPad Mini'
    elif '아이패드에어' in name or 'ipadair' in name: return 'iPad Air'
    elif '아이패드프로' in name or 'ipadpro' in name: return 'iPad Pro'
    elif '아이패드' in name or 'ipad' in name: return 'iPad'
    elif '맥북에어' in name or 'macbookair' in name: return 'MacBook Air'
    elif '맥북프로' in name or 'macbookpro' in name: return 'MacBook Pro'
    elif '맥북' in name or 'macbook' in name: return 'MacBook'
    elif '아이폰' in name or 'iphone' in name: return 'iPhone'
    elif '애플워치' in name or 'applewatch' in name: return 'Apple Watch'
    else: return '기타'

try:
    df = pd.read_excel(full_path)
    
    # 1. 숫자 및 사양 데이터 정제
    for col in ['release_price', 'price_sell', 'price_buy', 'device_age_months', '배터리_사이클']:
        if col in df.columns:
            df[col] = df[col].apply(clean_numeric)
    
    df['RAM_num'] = df['RAM'].apply(extract_number)
    df['SSD_num'] = df['SSD'].apply(extract_number)
    df['category'] = df['name_buy'].apply(classify_category_ko)

    # 2. 계층형 DB 구축
    hierarchical_db = {}
    for cat in df['category'].unique():
        cat_df = df[df['category'] == cat]
        hierarchical_db[cat] = cat_df.groupby('name_buy')['release_price'].median().to_dict()

    # 3. 학습용 데이터 확정 (결측치 제거)
    df = df.dropna(subset=['release_price', 'price_sell', 'price_buy', 'category']).copy()
    le = LabelEncoder()
    df['category_encoded'] = le.fit_transform(df['category'])

    # ============================================================
    # [SECTION 2] 인공지능 모델 학습 (S급 특화 버전)
    # ============================================================

    # [수정된 모델 1] S급(정상) 데이터 전용 시세 예측 모델
    status_cols = [
        '구매_경로', '라이트닝케이블', '배터리_사이클', '버튼_동작_iPad,_iPhone,_Galaxy_',
        '변색', '사설업체_수리이력', '생활_기스', '생활기스__for_Acc_', '액정_상태',
        '작동_여부', '전원_케이블_노트북_', '전원_케이블_모바일_', '제품_박스_유무',
        '제품_박스_유무_for_Acc_', '제품_박스_유무_iMac_', '찍힘_깨짐',
        '찍힘_깨짐__for_Acc_', '컨트롤러_작동_유무'
    ]
    existing_status_cols = [c for c in status_cols if c in df.columns]
    
    # S급 데이터 필터링
    is_clean = (df[existing_status_cols].sum(axis=1) == 0)
    s_grade_df = df[is_clean].copy()

    # 기존 rf_sell 이름을 그대로 사용하여 인터페이스 연동
    features_sell = ['category_encoded', 'device_age_months', 'release_price']
    X_s = s_grade_df[features_sell]
    y_s = s_grade_df['price_sell']
    X_s_train, X_s_test, y_s_train, y_s_test = train_test_split(X_s, y_s, test_size=0.2, random_state=42)
    
    rf_sell = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_sell.fit(X_s_train, y_s_train)

    # [모델 2] 매입가 예측 (전체 데이터 활용 유지 - 사양 기반)
    features_buy = ['category_encoded', 'device_age_months', 'release_price', 'RAM_num', 'SSD_num', '배터리_사이클']
    df['배터리_사이클'] = df['배터리_사이클'].fillna(0)
    X_b = df[features_buy]
    y_b = df['price_buy']
    X_b_train, X_b_test, y_b_train, y_b_test = train_test_split(X_b, y_b, test_size=0.2, random_state=42)
    
    rf_buy = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_buy.fit(X_b_train, y_b_train)

    # 성적표 출력
    print("\n" + "="*55)
    print(f"🤖 S급 특화 시세 예측 모델 구축 완료")
    print(f"- 사용된 S급 데이터: {len(s_grade_df)}대 (전체 {len(df)}대 중)")
    print(f"- 시세 모델(S급) R²: {r2_score(y_s_test, rf_sell.predict(X_s_test)):.3f}")
    print(f"- 매입 모델(전체) R²: {r2_score(y_b_test, rf_buy.predict(X_b_test)):.2f}")
    print("="*55)

    # ============================================================
    # [SECTION 3] 인터페이스 실행 (기존 코드와 100% 호환)
    # ============================================================
    def run_system():
        print("\n🍎 Apple 중고 통합 예측 시스템 (S급 시세 기준)")
        cats = sorted(list(hierarchical_db.keys()))
        print("\n▶ [STEP 1] 카테고리 선택:")
        for i, c in enumerate(cats): print(f"  {i+1}. {c}")
        
        try:
            c_idx = int(input("번호: ")) - 1
            sel_cat = cats[c_idx]
            prods = sorted(list(hierarchical_db[sel_cat].keys()))
            print(f"\n▶ [STEP 2] {sel_cat} 상세 모델 선택:")
            for i, p in enumerate(prods): print(f"  {i+1}. {p}")
            p_idx = int(input("번호: ")) - 1
            sel_prod = prods[p_idx]
            
            auto_price = hierarchical_db[sel_cat][sel_prod]
            print(f"\n✅ 모델: {sel_prod} (출시가: {int(auto_price):,}원)")
            qtr = int(input("▶ 사용 기간 (분기 단위): "))
            ram = input("▶ RAM 용량 (예: 8GB): ")
            ssd = input("▶ SSD 용량 (예: 256GB): ")
            bat = int(input("▶ 배터리 상태 (0:정상 ~ 5:교체필요): "))

            c_enc = le.transform([sel_cat])[0]
            months = (qtr * 3) - 1
            r_n, s_n = extract_number(ram), extract_number(ssd)

            # S급 모델(rf_sell)로 예측
            p_sell = rf_sell.predict(pd.DataFrame([[c_enc, months, auto_price]], columns=features_sell))[0]
            p_buy = rf_buy.predict(pd.DataFrame([[c_enc, months, auto_price, r_n, s_n, bat]], columns=features_buy))[0]

            print("\n" + "✨" * 20)
            print(f"📊 최종 분석 결과 (S급 시세 기반)")
            print(f"💡 예상 판매 시세 : {int(round(p_sell, -2)):,} 원")
            print(f"💰 최종 매입 제안 : {int(round(p_buy, -2)):,} 원")
            print(f"📢 예상 유통 수익 : {int(round(p_sell - p_buy, -2)):,} 원")
            print(f"📈 예상 보존율    : {round((p_sell / auto_price) * 100, 1)}%")
            print("✨" * 20 + "\n")
        except Exception as e:
            print(f"입력 오류: {e}")

    run_system()

except Exception as e:
    print(f"❌ 오류 발생: {e}")


🤖 S급 특화 시세 예측 모델 구축 완료
- 사용된 S급 데이터: 706대 (전체 1008대 중)
- 시세 모델(S급) R²: 0.713
- 매입 모델(전체) R²: 0.82

🍎 Apple 중고 통합 예측 시스템 (S급 시세 기준)

▶ [STEP 1] 카테고리 선택:
  1. Apple Watch
  2. MacBook
  3. MacBook Air
  4. MacBook Pro
  5. iPad
  6. iPad Air
  7. iPad Mini
  8. iPad Pro
  9. iPhone
  10. 기타


번호:  1



▶ [STEP 2] Apple Watch 상세 모델 선택:
  1. 애플 워치 38mm Aluminum GPS Series3
  2. 애플 워치 40mm Aluminum GPS SE
  3. 애플 워치 40mm Aluminum GPS Series4
  4. 애플 워치 40mm Aluminum GPS Series5
  5. 애플 워치 40mm Aluminum LTE SE
  6. 애플 워치 40mm Aluminum LTE Series5
  7. 애플 워치 40mm Aluminum LTE Series6
  8. 애플 워치 40mm Stainless LTE Series5
  9. 애플 워치 40mm Stainless LTE Series6
  10. 애플 워치 41mm Aluminum GPS Series7
  11. 애플 워치 41mm Aluminum GPS Series8
  12. 애플 워치 42mm Aluminum GPS Series3 스페이스그레이
  13. 애플 워치 42mm Aluminum LTE Series3
  14. 애플 워치 42mm Aluminum Series2
  15. 애플 워치 44mm Aluminum GPS SE
  16. 애플 워치 44mm Aluminum GPS Series4
  17. 애플 워치 44mm Aluminum GPS Series5 Nike
  18. 애플 워치 44mm Aluminum GPS Series5 스페이스그레이
  19. 애플 워치 44mm Aluminum GPS Series6
  20. 애플 워치 44mm Aluminum LTE SE
  21. 애플 워치 44mm Aluminum LTE Series4
  22. 애플 워치 44mm Aluminum NIKE GPS Series6
  23. 애플 워치 44mm Stainless LTE Series4
  24. 애플 워치 45mm Aluminum GPS Series7
  25. 애플 워치 45mm Aluminum GPS Series8
  26. 애플 워치 45mm Sta

번호:  1



✅ 모델: 애플 워치 38mm Aluminum GPS Series3 (출시가: 429,000원)


▶ 사용 기간 (분기 단위):  1
▶ RAM 용량 (예: 8GB):  1
▶ SSD 용량 (예: 256GB):  1
▶ 배터리 상태 (0:정상 ~ 5:교체필요):  1



✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨
📊 최종 분석 결과 (S급 시세 기반)
💡 예상 판매 시세 : 393,500 원
💰 최종 매입 제안 : 320,900 원
📢 예상 유통 수익 : 72,600 원
📈 예상 보존율    : 91.7%
✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨✨



In [ ]:
# ============================================================
# [SECTION 2-2] 카테고리별 개별 모델 학습 및 성능 분석
# ============================================================

category_models = {} # 카테고리별 모델 저장소
category_results = [] # 결과 비교표 저장

print("\n" + "📊" * 20)
print("🔍 [강사님 제언] 카테고리별 개별 모델 성능 분석")

for cat in df['category'].unique():
    # 1. 해당 카테고리 데이터만 추출
    cat_df = df[df['category'] == cat].copy()
    
    # 데이터가 너무 적으면 학습이 불가능하므로 최소 개수 확인 (예: 10개 이상)
    if len(cat_df) < 10:
        continue
        
    # 2. 데이터 분할 (카테고리 내에서 8:2)
    X_cat = cat_df[features_sell]
    y_cat = cat_df['price_sell']
    X_train, X_test, y_train, y_test = train_test_split(X_cat, y_cat, test_size=0.2, random_state=42)
    
    # 3. 개별 모델 학습
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # 4. 성능 측정
    r2 = r2_score(y_test, model.predict(X_test))
    mape = mean_absolute_percentage_error(y_test, model.predict(X_test)) * 100
    
    # 5. 모델 저장 및 리포트 데이터 수집
    category_models[cat] = model
    category_results.append({
        '카테고리': cat,
        '데이터수': len(cat_df),
        '개별모델 R²': round(r2, 3),
        '오차율(%)': round(mape, 1)
    })

# 결과 출력
res_df = pd.DataFrame(category_results).sort_values(by='개별모델 R²', ascending=False)
print(res_df.to_string(index=False))
print(f"\n👉 전체 모델 R²: 0.67 vs 카테고리별 평균 R²: {res_df['개별모델 R²'].mean():.2f}")
print("📊" * 20 + "\n")

In [ ]:
# ============================================================
# [SECTION 2-3] 하이브리드 모델 전략 (에이스 카테고리 선별)
# ============================================================

ace_category_models = {}
hybrid_results = []

# 기준 설정: 데이터 50개 이상 & R2 0.7 이상인 것만 '에이스'로 인정
MIN_SAMPLES = 50
MIN_R2 = 0.7

print("\n" + "🌟" * 20)
print("🚀 [하이브리드 전략] 에이스 카테고리 선별 결과")

for cat in df['category'].unique():
    cat_df = df[df['category'] == cat].copy()
    
    # 데이터가 너무 적으면 패스
    if len(cat_df) < MIN_SAMPLES:
        continue
        
    X_cat = cat_df[features_sell]
    y_cat = cat_df['price_sell']
    X_train, X_test, y_train, y_test = train_test_split(X_cat, y_cat, test_size=0.2, random_state=42)
    
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    r2 = r2_score(y_test, model.predict(X_test))
    
    # 성능 기준 미달이면 패스 (에바인 값 거르기)
    if r2 < MIN_R2:
        continue
        
    # 기준 통과한 '에이스'들만 저장
    ace_category_models[cat] = model
    hybrid_results.append({
        '카테고리': cat,
        '데이터수': len(cat_df),
        '에이스모델 R²': round(r2, 3)
    })

ace_df = pd.DataFrame(hybrid_results).sort_values(by='에이스모델 R²', ascending=False)
print(ace_df.to_string(index=False))
print(f"\n✅ 총 {len(ace_df)}개의 카테고리가 '에이스 전용 모델'로 선정되었습니다.")
print("🌟" * 20 + "\n")

In [ ]:
# ============================================================
# [SECTION 2-4] S급(모든 상태 정상) 데이터 한정 성능 분석
# ============================================================

# 1. '정상' 조건 리스트 (사용자님이 주신 컬럼들)
status_cols = [
    '구매_경로', '라이트닝케이블', '배터리_사이클', '버튼_동작_iPad,_iPhone,_Galaxy_',
    '변색', '사설업체_수리이력', '생활_기스', '생활기스__for_Acc_', '액정_상태',
    '작동_여부', '전원_케이블_노트북_', '전원_케이블_모바일_', '제품_박스_유무',
    '제품_박스_유무_for_Acc_', '제품_박스_유무_iMac_', '찍힘_깨짐',
    '찍힘_깨짐__for_Acc_', '컨트롤러_작동_유무'
]

# 2. 모든 상태 컬럼이 0(정상)인 데이터만 추출
# (주의: 데이터 전처리 단계에서 이미 숫자로 변환되어 있어야 합니다)
clean_df = df.copy()

# 모든 상태 컬럼의 합이 0인 로우만 필터링 (결함이 하나도 없는 데이터)
# 만약 데이터가 0이 아닌 다른 방식(예: '정상')으로 되어 있다면 그에 맞게 수정 필요
is_clean = (clean_df[status_cols].sum(axis=1) == 0)
s_grade_df = clean_df[is_clean].copy()

print("\n" + "✨" * 20)
print(f"🎯 [S급 데이터 한정] 성능 분석 리포트")
print(f"- 전체 데이터 수: {len(df)}대")
print(f"- S급(정상) 데이터 수: {len(s_grade_df)}대")

if len(s_grade_df) >= 20: # 데이터가 어느 정도 있을 때만 실행
    # 3. 데이터 분할 및 학습
    X_s_grade = s_grade_df[features_sell]
    y_s_grade = s_grade_df['price_sell']
    X_train, X_test, y_train, y_test = train_test_split(X_s_grade, y_s_grade, test_size=0.2, random_state=42)
    
    s_model = RandomForestRegressor(n_estimators=100, random_state=42)
    s_model.fit(X_train, y_train)
    
    # 4. 성능 측정
    r2 = r2_score(y_test, s_model.predict(X_test))
    mape = mean_absolute_percentage_error(y_test, s_model.predict(X_test)) * 100
    
    print(f"- S급 모델 결정계수(R²): {r2:.3f}")
    print(f"- S급 모델 오차율(MAPE): {mape:.1f}%")
    print("\n👉 결함 변수를 배제하고 '제품 본연의 가치'만 평가했을 때의 정확도입니다.")
else:
    print("❌ S급 데이터가 너무 적어 분석이 불가능합니다.")
print("✨" * 20 + "\n")

In [ ]:
# ============================================================
    # [추가] K-Fold 교차 검증 (발표용 신뢰도 증명)
    # ============================================================
from sklearn.model_selection import cross_val_score, KFold

print("\n" + "🎲" * 20)
print("🎯 실전 검증: K-Fold 교차 검증 수행 (5-Fold)")
    
# 교차 검증용 설정 (데이터를 5개로 나눠 5번 교차 테스트)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
# 1. 시세 모델(rf_sell) 검증
cv_scores_sell = cross_val_score(rf_sell, X_s, y_s, cv=kf, scoring='r2')
    
# 2. 매입 모델(rf_buy) 검증
cv_scores_buy = cross_val_score(rf_buy, X_b, y_b, cv=kf, scoring='r2')

print(f"📈 [시세 모델] 평균 R²: {cv_scores_sell.mean():.2f} (±{cv_scores_sell.std():.2f})")
print(f"📈 [매입 모델] 평균 R²: {cv_scores_buy.mean():.2f} (±{cv_scores_buy.std():.2f})")
print("👉 '평균 점수'가 높고 '편차(±)'가 낮을수록 어떤 데이터에도 강한 모델입니다.")
print("🎲" * 20)

In [ ]:
from sklearn.model_selection import ShuffleSplit, cross_val_score

# 1. 설정: 데이터를 10번 무작위로 섞어서, 매번 20%를 시험용으로 사용
rs = ShuffleSplit(n_splits=10, test_size=0.2, random_state=42)

print("\n" + "🎲" * 20)
print("🎯 [심화 검증] Random Subsampling (10회 반복)")

# 2. 시세 모델(rf_sell) 검증
rs_scores_sell = cross_val_score(rf_sell, X_s, y_s, cv=rs, scoring='r2')

# 3. 매입 모델(rf_buy) 검증
rs_scores_buy = cross_val_score(rf_buy, X_b, y_b, cv=rs, scoring='r2')

print(f"📈 [시세 모델] 무작위 10회 평균 R²: {rs_scores_sell.mean():.2f} (±{rs_scores_sell.std():.2f})")
print(f"📈 [매입 모델] 무작위 10회 평균 R²: {rs_scores_buy.mean():.2f} (±{rs_scores_buy.std():.2f})")
print("👉 K-Fold보다 더 무작위성이 강한 검증에서도 안정적인 성능을 보입니다.")
print("🎲" * 20 + "\n")

In [ ]:
# 1. 모델 설정 변경 (oob_score=True 추가)
# 이 설정은 학습할 때 '나를 공부하는 데 쓰지 않은 나머지 데이터'로 즉석 시험을 보라는 뜻입니다.
rf_sell_boot = RandomForestRegressor(n_estimators=100, oob_score=True, random_state=42)
rf_buy_boot = RandomForestRegressor(n_estimators=100, oob_score=True, random_state=42)

# 2. 전체 데이터로 학습 (별도의 test set 없이도 내부에서 시험을 치릅니다)
rf_sell_boot.fit(X_s, y_s)
rf_buy_boot.fit(X_b, y_b)

print("\n" + "🥾" * 20)
print("🎯 [심화 검증] Bootstrapping (OOB Score)")

# 3. 결과 출력
print(f"📈 [시세 모델] OOB 결정계수(R²): {rf_sell_boot.oob_score_:.2f}")
print(f"📈 [매입 모델] OOB 결정계수(R²): {rf_buy_boot.oob_score_:.2f}")
print("👉 OOB 점수는 별도의 시험지 없이도 모델 내부에서 엄격하게 검증된 수치입니다.")
print("🥾" * 20 + "\n")

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score

# 1. 안쪽 루프(Inner Loop): 최적의 하이퍼파라미터 찾기 설정
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

# 2. 바깥쪽 루프(Outer Loop)용 모델 설정
# 안쪽에서 GridSearchCV로 최적의 파라미터를 찾으면서 5-Fold를 돌립니다.
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

# GridSearchCV 객체 생성
rf_grid = GridSearchCV(estimator=RandomForestRegressor(random_state=42), 
                       param_grid=param_grid, cv=inner_cv, scoring='r2')

print("\n" + "🪆" * 20)
print("🎯 [최종 검증] Nested Cross-Validation (이중 중첩 검증)")
print("기다려주세요... 모델이 스스로 최적의 설정을 찾으며 검증 중입니다.")

# 3. 중첩 교차 검증 실행 (매입 모델 기준)
nested_score = cross_val_score(rf_grid, X_b, y_b, cv=outer_cv)

print(f"📈 [매입 모델] Nested CV 평균 R²: {nested_score.mean():.2f}")
print(f"📈 [매입 모델] 개별 Fold 점수: {nested_score}")
print("👉 '시험 문제'를 보고 '공부 방법'을 정하는 반칙(Leakage)을 완벽히 차단했습니다.")
print("🪆" * 20 + "\n")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# [모델 1] 시세 예측 성적표
y_s_pred = rf_sell.predict(X_s_test)
mae_sell = mean_absolute_error(y_s_test, y_s_pred)
mape_sell = mean_absolute_percentage_error(y_s_test, y_s_pred) * 100

# [모델 2] 매입가 예측 성적표
y_b_pred = rf_buy.predict(X_b_test)
mae_buy = mean_absolute_error(y_b_test, y_b_pred)
mape_buy = mean_absolute_percentage_error(y_b_test, y_b_pred) * 100

print("\n" + "📊" * 20)
print("🎯 [모델 성능 정밀 성적표]")
print(f"1. 시세 예측 모델 (Sell Price)")
print(f"   - 실제 금액 오차(MAE): 평균 {int(mae_sell):,}원")
print(f"   - 백분율 오차(MAPE): {mape_sell:.2f}%")

print(f"\n2. 매입가 예측 모델 (Buy Price)")
print(f"   - 실제 금액 오차(MAE): 평균 {int(mae_buy):,}원")
print(f"   - 백분율 오차(MAPE): {mape_buy:.2f}%")
print("👉 MAPE가 10% 미만이면 '아주 우수한' 모델로 평가받습니다.")
print("📊" * 20 + "\n")